# 03 · Data Cleaning

**Project:** Enterprise HR AI  
**Rule:** Every cleaning decision is printed and logged. No silent changes.

---

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

RAW  = os.path.join('..', 'data', 'raw')
PROC = os.path.join('..', 'data', 'processed')
os.makedirs(PROC, exist_ok=True)

print('RAW  :', os.path.abspath(RAW))
print('PROC :', os.path.abspath(PROC))

RAW  : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\raw
PROC : C:\Users\ASUS\Desktop\enterprise_hr_ai\data\processed


In [2]:
attrition_raw = pd.read_csv(os.path.join(RAW, 'employee_attrition.csv'))
hr_raw        = pd.read_csv(os.path.join(RAW, 'Cleaned_HR_Data_Analysis.csv'))

print(f'employee_attrition.csv        : {attrition_raw.shape[0]:,} rows x {attrition_raw.shape[1]} cols')
print(f'Cleaned_HR_Data_Analysis.csv  : {hr_raw.shape[0]:,} rows x {hr_raw.shape[1]} cols')

# Work on copies so raw is never mutated
attrition = attrition_raw.copy()
hr        = hr_raw.copy()

employee_attrition.csv        : 1,470 rows x 35 cols
Cleaned_HR_Data_Analysis.csv  : 2,845 rows x 28 cols


---
## Step 0 · Age=17 Investigation — Employee ID 1743 & 2038

Before any cleaning, we surface the full rows for the two offending records from
`Cleaned_HR_Data_Analysis.csv` so we can assess whether a correction is unambiguous.

In [3]:
FLAGGED_IDS = [1743, 2038]

for emp_id in FLAGGED_IDS:
    row = hr[hr['Employee ID'] == emp_id]
    print(f'\n{"="*70}')
    print(f'FULL ROW — Employee ID {emp_id}')
    print(f'{"="*70}')
    print(row.T.to_string())
    print()


FULL ROW — Employee ID 1743
                                               1228
Employee ID                                    1743
StartDate                                 30-Nov-18
Title                       Production Technician I
BusinessUnit                                    SVG
EmployeeStatus                               Active
EmployeeType                              Full-Time
PayZone                                      Zone C
EmployeeClassificationType                Temporary
DepartmentType                    Production       
Division                                   Splicing
DOB                                      03-06-2001
State                                            MA
GenderCode                                     Male
RaceDesc                                      Black
MaritalDesc                                  Single
Performance Score                       Fully Meets
Current Employee Rating                           3
Survey Date                        

In [4]:
import datetime

SURVEY_DATE_COL  = 'Survey Date'
DOB_COL          = 'DOB'
AGE_COL          = 'Age'
EXCLUDED_ROWS    = []
CORRECTIONS_LOG  = []

def parse_date(s):
    for fmt in ('%d-%m-%Y', '%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y',
                '%d-%b-%y', '%d-%b-%Y', '%b %d, %Y'):
        try:
            return datetime.datetime.strptime(str(s).strip(), fmt)
        except Exception:
            pass
    return None

def compute_age_from_dob(dob_str, ref_date_str):
    dob = parse_date(dob_str)
    ref = parse_date(ref_date_str)
    if dob is None or ref is None:
        return None
    age = (ref - dob).days // 365
    return int(age)

print('=== AGE=17 DECISION LOGIC ===')
print()

for emp_id in FLAGGED_IDS:
    idx  = hr.index[hr['Employee ID'] == emp_id][0]
    row  = hr.loc[idx]
    recorded_age = row[AGE_COL]

    dob_str    = row.get(DOB_COL, None)
    sdate_str  = row.get(SURVEY_DATE_COL, None)

    print(f'--- Employee ID {emp_id} ---')
    print(f'  Recorded Age : {recorded_age}')
    print(f'  DOB          : {dob_str}')
    print(f'  Survey Date  : {sdate_str}')

    computed_age = compute_age_from_dob(dob_str, sdate_str) if (dob_str and sdate_str) else None
    print(f'  Computed Age from DOB -> Survey Date: {computed_age}')

    if computed_age is not None and computed_age >= 18 and computed_age != recorded_age:
        print(f'  DECISION: Correct Age {recorded_age} -> {computed_age} '
              f'(unambiguous: derived from DOB={dob_str} and Survey Date={sdate_str})')
        CORRECTIONS_LOG.append({
            'Employee ID': emp_id, 'Column': AGE_COL,
            'Before': recorded_age, 'After': computed_age,
            'Justification': f'Age computed from DOB ({dob_str}) to Survey Date ({sdate_str})'
        })
        hr.at[idx, AGE_COL] = computed_age
    else:
        reason = (
            'DOB not present or unparseable' if computed_age is None
            else f'Computed age {computed_age} is also < 18 or equals recorded age — ambiguous'
        )
        print(f'  DECISION: EXCLUDE row — reason: {reason}')
        exc_row = row.to_dict()
        exc_row['exclusion_reason'] = reason
        EXCLUDED_ROWS.append(exc_row)
    print()

# Write exclusion log regardless of whether rows were excluded or corrected
if EXCLUDED_ROWS:
    exc_df = pd.DataFrame(EXCLUDED_ROWS)
    exc_path = os.path.join(PROC, 'excluded_rows_log.csv')
    exc_df.to_csv(exc_path, index=False)
    print(f'Excluded {len(EXCLUDED_ROWS)} row(s) -> logged to {exc_path}')
    # Drop excluded rows from hr
    excluded_ids = [r['Employee ID'] for r in EXCLUDED_ROWS]
    hr = hr[~hr['Employee ID'].isin(excluded_ids)].reset_index(drop=True)
    print(f'hr shape after exclusion: {hr.shape}')

if CORRECTIONS_LOG:
    print('\nCorrections applied:')
    for c in CORRECTIONS_LOG:
        print(f'  Employee ID {c["Employee ID"]}: {c["Column"]} {c["Before"]} -> {c["After"]}')
        print(f'  Justification: {c["Justification"]}')

=== AGE=17 DECISION LOGIC ===

--- Employee ID 1743 ---
  Recorded Age : 17
  DOB          : 03-06-2001
  Survey Date  : 20-04-2023
  Computed Age from DOB -> Survey Date: 21
  DECISION: Correct Age 17 -> 21 (unambiguous: derived from DOB=03-06-2001 and Survey Date=20-04-2023)

--- Employee ID 2038 ---
  Recorded Age : 17
  DOB          : 04-05-2001
  Survey Date  : 18-02-2023
  Computed Age from DOB -> Survey Date: 21
  DECISION: Correct Age 17 -> 21 (unambiguous: derived from DOB=04-05-2001 and Survey Date=18-02-2023)


Corrections applied:
  Employee ID 1743: Age 17 -> 21
  Justification: Age computed from DOB (03-06-2001) to Survey Date (20-04-2023)
  Employee ID 2038: Age 17 -> 21
  Justification: Age computed from DOB (04-05-2001) to Survey Date (18-02-2023)


In [5]:
print('=== RE-RUNNING V-HR-4 AGE ASSERT (from notebook 02) ===')
bad_age_after = hr[(hr['Age'] < 18) | (hr['Age'] > 100)]
if len(bad_age_after) > 0:
    print(f'Offending rows still present ({len(bad_age_after)}):')
    print(bad_age_after[['Employee ID', 'Age']].to_string())
assert len(bad_age_after) == 0, (
    f'V-HR-4 still FAILED — {len(bad_age_after)} rows have Age outside [18, 100]'
)
print('V-HR-4 PASSED: Age range now clean, no values outside [18, 100]')

=== RE-RUNNING V-HR-4 AGE ASSERT (from notebook 02) ===
V-HR-4 PASSED: Age range now clean, no values outside [18, 100]


---
## Section 1 · Cleaning — employee_attrition.csv

In [6]:
print('--- 1.1  Strip whitespace from all object columns ---')
obj_cols_att = attrition.select_dtypes(include='object').columns.tolist()
for col in obj_cols_att:
    before_unique = attrition[col].dropna().unique()
    attrition[col] = attrition[col].str.strip()
    after_unique  = attrition[col].dropna().unique()
    changed = set(before_unique) - set(after_unique)
    if changed:
        print(f'  [{col}] stripped whitespace, removed variants: {changed}')
print(f'  Stripped {len(obj_cols_att)} object columns. No non-whitespace values changed.')

--- 1.1  Strip whitespace from all object columns ---
  Stripped 9 object columns. No non-whitespace values changed.


In [7]:
print('--- 1.2  Categorical unique-value audit (post-strip) ---')
CAT_THRESHOLD = 30
for col in attrition.select_dtypes(include='object').columns:
    uvals = sorted(attrition[col].dropna().unique().tolist())
    if len(uvals) <= CAT_THRESHOLD:
        print(f'  [{col}] ({len(uvals)} unique): {uvals}')

--- 1.2  Categorical unique-value audit (post-strip) ---
  [Attrition] (2 unique): ['No', 'Yes']
  [BusinessTravel] (3 unique): ['Non-Travel', 'Travel_Frequently', 'Travel_Rarely']
  [Department] (3 unique): ['Human Resources', 'Research & Development', 'Sales']
  [EducationField] (6 unique): ['Human Resources', 'Life Sciences', 'Marketing', 'Medical', 'Other', 'Technical Degree']
  [Gender] (2 unique): ['Female', 'Male']
  [JobRole] (9 unique): ['Healthcare Representative', 'Human Resources', 'Laboratory Technician', 'Manager', 'Manufacturing Director', 'Research Director', 'Research Scientist', 'Sales Executive', 'Sales Representative']
  [MaritalStatus] (3 unique): ['Divorced', 'Married', 'Single']
  [Over18] (1 unique): ['Y']
  [OverTime] (2 unique): ['No', 'Yes']


In [8]:
print('--- 1.3  Missing values ---')
null_counts = attrition.isnull().sum()
null_pct    = (null_counts / len(attrition) * 100).round(2)
missing_cols_att = null_counts[null_counts > 0]

if missing_cols_att.empty:
    print('  No missing values — nothing to impute.')
else:
    for col in missing_cols_att.index:
        pct = null_pct[col]
        dtype = attrition[col].dtype
        print(f'  [{col}]: {missing_cols_att[col]} missing ({pct}%)')
        if pd.api.types.is_numeric_dtype(dtype):
            med = attrition[col].median()
            attrition[col].fillna(med, inplace=True)
            print(f'    Strategy: MEDIAN imputation -> {med}')
        else:
            mode_val = attrition[col].mode()
            if len(mode_val) > 0:
                attrition[col].fillna(mode_val[0], inplace=True)
                print(f'    Strategy: MODE imputation -> {mode_val[0]}')
            else:
                attrition[col].fillna('Unknown', inplace=True)
                print(f'    Strategy: UNKNOWN fill (no mode available)')

--- 1.3  Missing values ---
  No missing values — nothing to impute.


In [9]:
print('--- 1.4  Duplicate rows ---')
n_before = len(attrition)
attrition.drop_duplicates(inplace=True)
attrition.reset_index(drop=True, inplace=True)
n_after  = len(attrition)
print(f'  Dropped {n_before - n_after} exact duplicate rows ({n_before} -> {n_after})')

--- 1.4  Duplicate rows ---
  Dropped 0 exact duplicate rows (1470 -> 1470)


In [10]:
print('--- 1.5  Dtype enforcement ---')

INT_COLS_ATT = [
    'Age', 'DailyRate', 'DistanceFromHome', 'Education',
    'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction',
    'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction',
    'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
    'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction',
    'StandardHours', 'StockOptionLevel', 'TotalWorkingYears',
    'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany',
    'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager'
]

for col in INT_COLS_ATT:
    if col in attrition.columns:
        old_dtype = attrition[col].dtype
        attrition[col] = pd.to_numeric(attrition[col], errors='coerce').astype('Int64')
        if str(old_dtype) != str(attrition[col].dtype):
            print(f'  [{col}]: {old_dtype} -> {attrition[col].dtype}')

print('  Dtype enforcement complete.')

--- 1.5  Dtype enforcement ---
  [Age]: int64 -> Int64
  [DailyRate]: int64 -> Int64
  [DistanceFromHome]: int64 -> Int64
  [Education]: int64 -> Int64
  [EmployeeCount]: int64 -> Int64
  [EmployeeNumber]: int64 -> Int64
  [EnvironmentSatisfaction]: int64 -> Int64
  [HourlyRate]: int64 -> Int64
  [JobInvolvement]: int64 -> Int64
  [JobLevel]: int64 -> Int64
  [JobSatisfaction]: int64 -> Int64
  [MonthlyIncome]: int64 -> Int64
  [MonthlyRate]: int64 -> Int64
  [NumCompaniesWorked]: int64 -> Int64
  [PercentSalaryHike]: int64 -> Int64
  [PerformanceRating]: int64 -> Int64
  [RelationshipSatisfaction]: int64 -> Int64
  [StandardHours]: int64 -> Int64
  [StockOptionLevel]: int64 -> Int64
  [TotalWorkingYears]: int64 -> Int64
  [TrainingTimesLastYear]: int64 -> Int64
  [WorkLifeBalance]: int64 -> Int64
  [YearsAtCompany]: int64 -> Int64
  [YearsInCurrentRole]: int64 -> Int64
  [YearsSinceLastPromotion]: int64 -> Int64
  [YearsWithCurrManager]: int64 -> Int64
  Dtype enforcement complete.


In [11]:
att_rows_before = attrition_raw.shape[0]
att_rows_after  = attrition.shape[0]
print(f'employee_attrition: {att_rows_before:,} rows -> {att_rows_after:,} rows  |  {attrition.shape[1]} cols')

employee_attrition: 1,470 rows -> 1,470 rows  |  35 cols


---
## Section 2 · Cleaning — Cleaned_HR_Data_Analysis.csv  (-> engagement_processed.csv)

In [12]:
print('--- 2.1  Strip whitespace from all object columns ---')
obj_cols_hr = hr.select_dtypes(include='object').columns.tolist()
whitespace_found = []
for col in obj_cols_hr:
    before = hr[col].dropna().tolist()
    hr[col] = hr[col].str.strip()
    after   = hr[col].dropna().tolist()
    changed_pairs = [(b, a) for b, a in zip(before, after) if b != a]
    if changed_pairs:
        whitespace_found.append((col, len(changed_pairs)))
        print(f'  [{col}]: stripped whitespace in {len(changed_pairs)} values')
        # Show unique dirty -> clean examples
        seen = set()
        for b, a in changed_pairs:
            key = (b, a)
            if key not in seen:
                print(f'    BEFORE: {repr(b)}')
                print(f'    AFTER : {repr(a)}')
                seen.add(key)
            if len(seen) >= 3:
                break
if not whitespace_found:
    print('  No whitespace variants found in any object column.')
print(f'  Stripped {len(obj_cols_hr)} object columns.')

--- 2.1  Strip whitespace from all object columns ---
  [Title]: stripped whitespace in 8 values
    BEFORE: 'Data Analyst '
    AFTER : 'Data Analyst'
  [DepartmentType]: stripped whitespace in 1910 values
    BEFORE: 'Production       '
    AFTER : 'Production'
  Stripped 20 object columns.


In [13]:
print('--- 2.2  Categorical unique-value audit (post-strip) ---')
CAT_THRESHOLD = 30
for col in hr.select_dtypes(include='object').columns:
    uvals = sorted(hr[col].dropna().unique().tolist())
    if len(uvals) <= CAT_THRESHOLD:
        print(f'  [{col}] ({len(uvals)} unique): {uvals}')

--- 2.2  Categorical unique-value audit (post-strip) ---
  [BusinessUnit] (10 unique): ['BPC', 'CCDR', 'EW', 'MSC', 'NEL', 'PL', 'PYZ', 'SVG', 'TNS', 'WBL']
  [EmployeeStatus] (2 unique): ['Active', 'Terminated']
  [EmployeeType] (3 unique): ['Contract', 'Full-Time', 'Part-Time']
  [PayZone] (3 unique): ['Zone A', 'Zone B', 'Zone C']
  [EmployeeClassificationType] (3 unique): ['Full-Time', 'Part-Time', 'Temporary']
  [DepartmentType] (6 unique): ['Admin Offices', 'Executive Office', 'IT/IS', 'Production', 'Sales', 'Software Engineering']
  [Division] (25 unique): ['Aerial', 'Billable Consultants', 'Catv', 'Corp Operations', 'Engineers', 'Executive', 'Field Operations', 'Fielders', 'Finance & Accounting', 'General - Con', 'General - Eng', 'General - Sga', 'Isp', 'People Services', 'Project Management - Con', 'Project Management - Eng', 'Safety', 'Sales & Marketing', 'Shop (Fleet)', 'Splicing', 'Technology / It', 'Underground', 'Wireless', 'Wireline Construction', 'Yard (Material Handlin

In [14]:
print('--- 2.3  Missing values ---')
null_counts_hr = hr.isnull().sum()
null_pct_hr    = (null_counts_hr / len(hr) * 100).round(2)
missing_cols_hr = null_counts_hr[null_counts_hr > 0]

if missing_cols_hr.empty:
    print('  No missing values — nothing to impute.')
else:
    for col in missing_cols_hr.index:
        pct   = null_pct_hr[col]
        dtype = hr[col].dtype
        print(f'  [{col}]: {missing_cols_hr[col]} missing ({pct}%)')
        if pd.api.types.is_numeric_dtype(dtype):
            med = hr[col].median()
            hr[col].fillna(med, inplace=True)
            print(f'    Strategy: MEDIAN imputation -> {med}')
        else:
            mode_val = hr[col].mode()
            if len(mode_val) > 0:
                hr[col].fillna(mode_val[0], inplace=True)
                print(f'    Strategy: MODE imputation -> {mode_val[0]}')
            else:
                hr[col].fillna('Unknown', inplace=True)
                print(f'    Strategy: UNKNOWN fill (no mode available)')

--- 2.3  Missing values ---
  No missing values — nothing to impute.


In [15]:
print('--- 2.4  Duplicate rows ---')
n_before_hr = len(hr)
hr.drop_duplicates(inplace=True)
hr.reset_index(drop=True, inplace=True)
n_after_hr  = len(hr)
print(f'  Dropped {n_before_hr - n_after_hr} exact duplicate rows ({n_before_hr} -> {n_after_hr})')

--- 2.4  Duplicate rows ---
  Dropped 0 exact duplicate rows (2845 -> 2845)


In [16]:
print('--- 2.5  Dtype enforcement ---')

INT_COLS_HR = ['Employee ID', 'Current Employee Rating',
               'Engagement Score', 'Satisfaction Score',
               'Work-Life Balance Score', 'Training Duration(Days)',
               'Age']

FLOAT_COLS_HR = ['Training Cost']

DATE_COLS_HR  = ['StartDate', 'DOB', 'Survey Date', 'Training Date']

for col in INT_COLS_HR:
    if col in hr.columns:
        old_dtype = hr[col].dtype
        hr[col] = pd.to_numeric(hr[col], errors='coerce').astype('Int64')
        if str(old_dtype) != str(hr[col].dtype):
            print(f'  [{col}]: {old_dtype} -> {hr[col].dtype}')

for col in FLOAT_COLS_HR:
    if col in hr.columns:
        old_dtype = hr[col].dtype
        hr[col] = pd.to_numeric(hr[col], errors='coerce')
        if str(old_dtype) != str(hr[col].dtype):
            print(f'  [{col}]: {old_dtype} -> {hr[col].dtype}')

for col in DATE_COLS_HR:
    if col in hr.columns:
        old_dtype = hr[col].dtype
        hr[col] = pd.to_datetime(hr[col], dayfirst=True, errors='coerce')
        print(f'  [{col}]: {old_dtype} -> {hr[col].dtype}')

print('  Dtype enforcement complete.')

--- 2.5  Dtype enforcement ---
  [Employee ID]: int64 -> Int64
  [Current Employee Rating]: int64 -> Int64
  [Engagement Score]: int64 -> Int64
  [Satisfaction Score]: int64 -> Int64
  [Work-Life Balance Score]: int64 -> Int64
  [Training Duration(Days)]: int64 -> Int64
  [Age]: int64 -> Int64


  [StartDate]: object -> datetime64[ns]


  [DOB]: object -> datetime64[ns]
  [Survey Date]: object -> datetime64[ns]


  [Training Date]: object -> datetime64[ns]
  Dtype enforcement complete.


In [17]:
hr_rows_before = hr_raw.shape[0]
hr_rows_after  = hr.shape[0]
print(f'Cleaned_HR_Data_Analysis: {hr_rows_before:,} rows -> {hr_rows_after:,} rows  |  {hr.shape[1]} cols')

Cleaned_HR_Data_Analysis: 2,845 rows -> 2,845 rows  |  28 cols


---
## Step 3 · Save to data/processed/

In [18]:
att_out  = os.path.join(PROC, 'employee_attrition_processed.csv')
hr_out   = os.path.join(PROC, 'engagement_processed.csv')

attrition.to_csv(att_out, index=False)
hr.to_csv(hr_out, index=False)

print(f'Saved: {att_out}')
print(f'       {attrition.shape[0]:,} rows x {attrition.shape[1]} cols')
print()
print(f'Saved: {hr_out}')
print(f'       {hr.shape[0]:,} rows x {hr.shape[1]} cols')
print()
print('Files in data/processed/:')
for f in sorted(os.listdir(PROC)):
    fpath = os.path.join(PROC, f)
    print(f'  {f}  ({os.path.getsize(fpath):,} bytes)')

Saved: ..\data\processed\employee_attrition_processed.csv
       1,470 rows x 35 cols

Saved: ..\data\processed\engagement_processed.csv
       2,845 rows x 28 cols

Files in data/processed/:
  employee_attrition_processed.csv  (227,974 bytes)
  engagement_processed.csv  (647,282 bytes)


---
## Cleaning Decision Log

*(Seed for `docs/data_relationships.md` — Step 4)*

---

### Before / After Row Counts

| File | Rows Before | Rows After | Delta |
|---|---|---|---|
| `employee_attrition.csv` | 1,470 | see output | see output |
| `Cleaned_HR_Data_Analysis.csv` → `engagement_processed.csv` | 2,845 | see output | see output |

---

### Decisions Made

#### Age=17 rows (Employee ID 1743, 2038 in Cleaned_HR_Data_Analysis.csv)
- Printed full rows and attempted to compute age from `DOB` and `Survey Date`.
- **If DOB-derived age was unambiguous (≥18 and ≠ recorded age):** corrected in-place with a printed before/after log.
- **If ambiguous or DOB unparseable:** excluded row; written to `data/processed/excluded_rows_log.csv` with `exclusion_reason` column.
- Re-ran V-HR-4 assert after treatment; it must pass before pipeline continues.

#### Whitespace stripping — both files
- Applied `.str.strip()` to **all** `object`-dtype columns in both files.
- Justification: Step 1 revealed `'Production       '` trailing-whitespace variants in `DepartmentType`.
  Whitespace is invisible in CSVs and creates phantom categories in groupby/merge operations.
- No semantic content was altered.

#### Missing values
- `employee_attrition.csv`: zero missing values — no imputation needed.
- `Cleaned_HR_Data_Analysis.csv`: zero missing values on required columns after Age=17 fix.
  Any missing values found were handled by: **MEDIAN** for numeric columns, **MODE** for categorical,
  **'Unknown'** if no mode exists. Each decision printed explicitly in Section 2.3.

#### Duplicates
- Exact duplicate rows dropped from both files (entire row identical across all columns).
- No partial-duplicate logic applied here — deduplication by key is a merge/join concern (Step 4).

#### Dtype enforcement
- `employee_attrition.csv`: 26 integer columns cast to `Int64` (nullable integer) to preserve NaN-safety.
- `Cleaned_HR_Data_Analysis.csv`: score/rating/age columns cast to `Int64`; `Training Cost` to `float64`;
  `StartDate`, `DOB`, `Survey Date`, `Training Date` parsed to `datetime64`.

#### Out-of-scope decisions (deferred)
- Skill-name normalization (`AWS` vs `Amazon Web Services`) — belongs to `essential_skills`/`software_skills` cleaning.
- Row-level join integrity between `employee_attrition` and `engagement_processed` — Step 4.
- `Employee_Performance_Dataset.csv` and `employee_performance_pro.csv` — not processed here
  (Step 1 recommended only `Cleaned_HR_Data_Analysis.csv` for processing based on ID-overlap evidence).

---

### Overlap note (from Step 1)

**Employee ID overlap between `employee_attrition.csv` (EmployeeNumber) and `engagement_processed.csv`
(Employee ID) is 49.7% (731/1,470). This validation confirms internal validity of each file only.
Row-level join validity is a Step 4 (data_relationships) concern.**